# V0.8
```

colabでの構成--

main.ipynb
       |___xxx.json
       |___my_plot.py
       |___【NEW】main.py  


```

* **[my_plot.py](https://github.com/kenkenkengo0421/Class_my_plt/blob/main/my_plot.py)**

In [1]:
!pip install -q kaggle-environments

# 提出関数👨‍🔧🔧

In [2]:
%%writefile main.py
import random

def step_toward(fx, fy, tx, ty, tiles):
    """
    最短ルートを計算
    目的地(tx, ty)に向かって安全に1歩進む

    1_X軸（東西）のチェック 目的地が右にあり、マップ右端(9)ではなく、右マスがLOCKEDでない場合：
      EASTを追加
    2_左（WEST）のチェック
    3_Y軸（南北）のチェック 目的地が下にあり、マップ下端(9)ではなく、下マスがLOCKEDでない場合：
      SOUTHを追加
    4_上（NORTH）のチェック
    5_目的地に近づく安全なルートがある場合：
      その中からランダムに1歩選ぶ（斜め移動時の角への引っかかりを防ぐため）
    6_目的地に近づけない場合：
     （障害物に塞がれている等）は、とりあえず移動可能な安全なマスを探す
    7_動ける場所があればランダム移動、完全に閉じ込められていればPASSを返す
    """
    candidates = []
    #1
    if fx < tx and fx < 9 and tiles[fy][fx + 1] != "LOCKED":
        candidates.append("EAST")
    #2
    elif fx > tx and fx > 0 and tiles[fy][fx - 1] != "LOCKED":
        candidates.append("WEST")
    #3
    if fy < ty and fy < 9 and tiles[fy + 1][fx] != "LOCKED":
        candidates.append("SOUTH")
    #4
    elif fy > ty and fy > 0 and tiles[fy - 1][fx] != "LOCKED":
        candidates.append("NORTH")
    #5
    if candidates:
        return random.choice(candidates)

    valid_dirs = []
    #6
    if fy > 0 and tiles[fy - 1][fx] != "LOCKED": valid_dirs.append("NORTH")
    if fy < 9 and tiles[fy + 1][fx] != "LOCKED": valid_dirs.append("SOUTH")
    if fx < 9 and tiles[fy][fx + 1] != "LOCKED": valid_dirs.append("EAST")
    if fx > 0 and tiles[fy][fx - 1] != "LOCKED": valid_dirs.append("WEST")

    #7
    return random.choice(valid_dirs) if valid_dirs else "PASS"


def find_target_tile(tiles, fx, fy, have_seeds, day, excluded_coords=None):
    """
    1_除外座標の指定がない（None）場合
        エラーを防ぐため空のセット（集合）を作成
    2_現在地が「処理をスキップする座標」に含まれていたら次のマスへ
    3_マスが通行不能・編集不可（LOCKED）ならスキップ
    4_雑草（WEED）が生えている場合
        草むしり（clear）候補として座標を追加
    5_作物の苗・植物（PLANT）が植えられている場合
        作物が植えられてからの経過日数を計算（現在の日数 - 植えた日数）
        作物の名前（種類）を取得。データがなければデフォルトで小麦（WHEAT）
        収穫可能になるまでの日数を設定（メロンなら10日、それ以外は2日）
    6_成長日数が収穫日数を過ぎている場合
        収穫（harvest）候補として座標を追加
    7_まだ収穫できず、かつ今日まだ水やりをしていない場合
        水やり（water）候補として座標を追加
    8_何も植えられていない空き地（None）かつ、種を持っている場合
        種まき（plant）候補として座標を追加
    9_作業候補が1つもみつからなかった（リストが空の）場合、None
    """

    if excluded_coords is None:
        excluded_coords = set()

    candidates = []

    #1
    for y in range(len(tiles)):
        for x in range(len(tiles[0])):
    #2
            if (x, y) in excluded_coords:
                continue

            t = tiles[y][x]
    #3
            if t == "LOCKED": continue
    #4
            if isinstance(t, dict) and t.get("kind") == "WEED":
                candidates.append((x, y, "clear"))
    #5
            elif isinstance(t, dict) and t.get("kind") == "PLANT":
                crop_age = day - t.get("planted_day", day)
                crop_name = t.get("crop", "WHEAT")
                harvest_age = 10 if crop_name == "MELON" else 2
    #6
                if crop_age >= harvest_age:
                    candidates.append((x, y, "harvest"))
    #7
                elif not t.get("watered_today", True):
                    candidates.append((x, y, "water"))
    #8
            elif t is None and have_seeds:
                candidates.append((x, y, "plant"))
    #9
    if not candidates:
        return None

    """
    仕事の優先順位を定義
    草むしり(0) ＞ 収穫(1) ＞ 水やり(2) ＞ 種まき(3) の順に優先される
    候補リストを「優先度」と「現在地からの距離」の2つの基準で並び替える
    一番条件の良い仕事場の座標(X, Y)を返す
    """

    priority = {"clear": 0, "harvest": 1, "water": 2, "plant": 3}
    candidates.sort(key=lambda c: (priority[c[2]], abs(c[0] - fx) + abs(c[1] - fy)))
    return candidates[0][0], candidates[0][1]




def agent(obs, config):
    """
    main
    """
    player = obs["player"]
    me = obs["farms"][player]
    private = obs["private"]

    fx, fy = me["farmer"]
    tiles = me["tiles"]
    tile = tiles[fy][fx]

    # 辞書にキーが無い場合のエラーを防ぐ
    money = me.get("money", 0)
    seeds = private.get("seeds", {})
    shed = private.get("shed", {})
    day = obs.get("day", 0)
    step = obs.get("step", 0)





    # ========================================
    # 1. 市場での売買・雇用ロジック
    # ========================================

    """
    1_お金が500以上の場合
        メロンの種を持ってなければ購入
        否で小麦の種を持ってなくてお金が10以上の場合小麦の種購入
    2_小麦の在庫があれば売却
    3_710ステップ以降になればメロン持ってれば売却
    　  710以前なら5つ以上メロン持ってれば売却
    4_人員：初日から4人、260日以降から計10人
    5_「現在の総人数」より「target_people」が多くてお金が200以上ある場合
    　　"HIRE"雇う
    6_ 土地：170日以降から計50マス（2象限）、260日以降から計75マス（3象限）
    7_今ある土地の数がtarget_quads_countよりも少ない、かつmoney（所持金）が 1500 以上ある場合
    　　"BUY_LAND"土地購入


    """
    #労働者たちの行動リスト:"hands"
    current_hands = me.get("hands", [])
    market = []


    wheat_seeds = seeds.get("WHEAT", 0) #小麦"WHEAT"の種
    melon_seeds = seeds.get("MELON", 0) #メロン"MELON"の種
    wheat_in_shed = shed.get("WHEAT", 0) #在庫数
    melon_in_shed = shed.get("MELON", 0)

    #ゲームに参加している「現在の総人数」
    total_people = 1 + len(current_hands)

    #NW（北西）、NE（北東）、SW（南西）、SE（南東）
    unlocked_quads = me.get("unlocked_quadrants", ["NW"])

    #1
    if money >= 500:
        if melon_seeds == 0:
            market.append(["BUY_SEED", "MELON", 1])
    else:
        if wheat_seeds == 0 and money >= 10:
            market.append(["BUY_SEED", "WHEAT", 1])
    #2
    if wheat_in_shed > 0:
        market.append(["SELL", "WHEAT", wheat_in_shed])
    #3
    if step >= 710:
        if melon_in_shed > 0:
            market.append(["SELL", "MELON", melon_in_shed])
    elif melon_in_shed >= 5:
        market.append(["SELL", "MELON", melon_in_shed])

    #4
    target_people = 10 if day >= 260 else 4

    #5
    if total_people < target_people and money >= 200:
        market.append(["HIRE"])

    #6
    target_quads_count = 1
    if day >= 260:
        target_quads_count = 3
    elif day >= 170:
        target_quads_count = 2

    #7
    if len(unlocked_quads) < target_quads_count and money >= 1500:
        market.append(["BUY_LAND"])



    # ========================================
    # 2. メイン農家のアクション
    # ========================================

    """
    1_足元が空き地の場合
    2_種を持ってたら植える
    3_足元に雑草"WEED"がある場合
      "DIG"抜く
    4_足元に植物"PLANT"がある場合
      今の日数 - 植えた日数 を計算して、「この作物は植えてから何日経ったか
      作物の「種類」を確認
    5_足元の作物がメロンなら10日待つ小麦なら2日待つ
    6_作物が育った日数（crop_age）が、収穫に必要な日数（harvest_age）に達している場合
    7_今日の水やりがまだなら水やり
    """


    farmer_action = None

    #1
    if tile is None:

    #2
        if melon_seeds > 0:
            farmer_action = ["PLANT", "MELON"]
        elif wheat_seeds > 0:
            farmer_action = ["PLANT", "WHEAT"]

    #3
    elif isinstance(tile, dict) and tile.get("kind") == "WEED":
        farmer_action = ["DIG"]

    #4
    elif isinstance(tile, dict) and tile.get("kind") == "PLANT":
        crop_age = day - tile.get("planted_day", day)
        crop_name = tile.get("crop", "WHEAT")
    #5
        harvest_age = 10 if crop_name == "MELON" else 2

    #6
        if crop_age >= harvest_age:
            farmer_action = ["HARVEST"]
    #7
        elif not tile.get("watered_today", True):
            farmer_action = ["WATER"]

    # ========================================
    # 3. 農家の移動
    # ========================================

    """
    1_まだ今ターンの行動が決まっていない場合
      仕事を探す関数"find_target_tile()"
    2_目的地が見つかった場合
      目的地をロック
      目的地に向かって、上下左右どちらに1マス進めばいいかを計算
    3_目的地がどこにもない場合
      今ターンはその場から動かない
    4_今ターンの行動が決まっている場合
      他のキャラクターがこのマスに侵入したり作業したりするのを防ぐ
    """
    claimed_targets = set()

    #1
    if farmer_action is None:
        has_any_seed = (wheat_seeds > 0) or (melon_seeds > 0)
        target = find_target_tile(tiles, fx, fy, has_any_seed, day, claimed_targets)
    #2
        if target:
            claimed_targets.add((target[0], target[1]))
            move_dir = step_toward(fx, fy, target[0], target[1], tiles)
            farmer_action = [move_dir]
        else:
            farmer_action = ["PASS"]
    #4
    else:
        claimed_targets.add((fx, fy))

    # ========================================
    # 4. 労働者(hands)の行動
    # ========================================

    """
    1_足元の種類が"WEED"雑草の場合
    　"DIG"抜く
    2_作物の苗"PLANT"の場合
    　成長日数を計算し、収穫期を過ぎていればその場で収穫"HARVEST"
    　足元の作物がメロンなら10日待つ小麦なら2日待つ
    3_作物が育った日数（crop_age）が、収穫に必要な日数（harvest_age）に達している場合収穫"HARVEST"
    4_今日の水やりがまだなら水やり
    5_足元が空き地で種があれば、その場で種を植える
    6_足元に仕事がない場合,遠くの仕事場を探して1歩移動する
    7_足元で作業（CLEAR, HARVEST, WATER）を行う場合、その自分の現在地もロックに含める

    """

    hands_actions = []

    # 既存の変数（小麦やメロンの種を持っているか）を利用して、従業員も種まきができるようにする
    has_any_seed = (wheat_seeds > 0) or (melon_seeds > 0)
    # 労働者の行動ループ
    for hand in current_hands:
        hx, hy = hand
        hand_tile = tiles[hy][hx]
        hand_action = None

    #1
        if isinstance(hand_tile, dict) and hand_tile.get("kind") == "WEED":
            hand_action = ["DIG"]
    #2
        elif isinstance(hand_tile, dict) and hand_tile.get("kind") == "PLANT":
            crop_age = day - hand_tile.get("planted_day", day)
            crop_name = hand_tile.get("crop", "WHEAT")
            harvest_age = 10 if crop_name == "MELON" else 2
    #3
            if crop_age >= harvest_age:
                hand_action = ["HARVEST"]
    #4
            elif not hand_tile.get("watered_today", True):
                hand_action = ["WATER"]
    #5
        elif hand_tile is None and has_any_seed:
            hand_action = ["PLANT", "WHEAT"]


    #6
        if hand_action is None:
            target = find_target_tile(tiles, hx, hy, has_any_seed, day, claimed_targets)
            if target:
                claimed_targets.add((target[0], target[1]))  # 目的地をロック
                move_dir = step_toward(hx, hy, target[0], target[1], tiles)
                hand_action = [move_dir]
            else:
                hand_action = ["PASS"]
    #7
        else:
            claimed_targets.add((hx, hy))

        hands_actions.append(hand_action)

    return {
        "farmer": farmer_action,
        "hands": hands_actions,
        "market": market
    }

Overwriting main.py


# テスト📝

In [6]:
from zoneinfo import ZoneInfo
import datetime
from kaggle_environments import make
from IPython.display import HTML



print(datetime.datetime.now(ZoneInfo("Asia/Tokyo")))


env = make("kaggriculture", configuration={"episodeSteps": 720}, debug=True)

env.run(["main.py", "random"])

final = env.steps[-1]
for i, s in enumerate(final):
    print(f"Player {i}: reward={s.reward}, status={s.status}")
"""
html_output = env.render(mode="html", width=500, height=500)
HTML(html_output)
"""

2026-08-07 00:02:25.003852+09:00
Player 0: reward=41911.0, status=DONE
Player 1: reward=0.0, status=DONE


'\nhtml_output = env.render(mode="html", width=500, height=500)\nHTML(html_output)\n'

# 分析📊



| 階層1 (トップレベル) | 階層2 (`steps`内) | 階層3 (詳細) | 記録されている内容 |
| :--- | :--- | :--- | :--- |
| `configuration` | | | ゲームの設定値（最大ターン数720、初期資金3000、ボードサイズなど） |
| `info` | | | 対戦プレイヤーの名前やシード値 |
| `rewards` | | | 最終的なスコア（所持金） |
| `steps` | | | ターンごとの全記録が入ったリスト（分析において最も重要） |
| | `action` | | そのターンにプレイヤーが送ったコマンド |
| | | `farmer` | メイン農家の行動（例: `["PLANT", "WHEAT"]` など） |
| | | `hands` | 労働者たちの行動リスト |
| | | `market` | 市場での売買・雇用アクション（例: `["BUY_SEED", "WHEAT", 1]`, `["HIRE"]` など） |
| | `observation` | | そのターンのゲーム内の全状況 |
| | | `step` / `day` / `hour` | 現在の進行度（何日目の何時間目か） |
| | | `market` | 市場の状況（`prices`: 現在の価格、`inventory`: 市場の在庫数） |
| | | `farms` | プレイヤー0と1（敵）の農場状態（`money`: 所持金、`tiles`: 10x10のマス目の状態、`farmer` / `hands`: 座標、`hires_today`: 今日の雇用人数） |
| | | `private` | 自分だけが見える非公開情報（`shed`: 小屋の収穫物在庫、`seeds`: 所持している種の数） |
| | | `town` | 町の状況（`unlocked_shops`: 解放されているショップのリスト） |
| | `reward` | | その時点でのスコア（所持金と同値） |
| | `status` | | エラー落ちしていないか（"ACTIVE" または "DONE"） |

In [4]:

import sys

sys.path.append('/kaggle/input/datasets/nagatakengo/kaggriculture-data-class')


import json
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.style
import seaborn as sns
import numpy as np
from my_plot import sns_line_s



def j_log(file):
    with open(file, 'r') as f:
        log = json.load(f)

    print(f"""
========================================================
json file {file} を読み込み完了...

""")

    my_id = 0
    found = False

    if "Agents" in log:
        for idx, agent in enumerate(log["Agents"]):
            if agent.get("Name") == "nk":
                my_id = idx
                found = True
                break

    if not found and "info" in log and "TeamNames" in log["info"]:
        team_names = log["info"]["TeamNames"]
        if "nk" in team_names:
            my_id = team_names.index("nk")
            found = True
    # デバッグ用の出力：本当に nk の ID が取れているか確認
    if found:
        print(f"プレイヤー 'nk' を Player {my_id} として認識しました")
    else:
        print(f"⚠️ 警告: ログの中に 'nk' が見つかりませんでした。強制的に Player 0 を使用します。")
        print(f"参考までに、このログのキー一覧: {list(log.keys())}")


    data_list = []

    for i, step in enumerate(log['steps']):
        if i == 0: continue

        # 観測データと市場データを取得
        my_obs = step[my_id]['observation']

        # 市場データ（市場価格は共通なので、my_obsから取っても同じ結果になります）
        market_prices = my_obs['market']['prices']

        #自分の所持金
        my_money = my_obs['farms'][my_id]['money']

        # 自分のアクション
        my_market_action = step[my_id]['action'].get('market', []) if 'action' in step[my_id] else []

        # 自分の小屋の在庫（これで確実に自分の在庫が取れます！）
        my_shed = my_obs['private']['shed']


        # データを1行にまとめる
        row = {

            'step': i,
            'p0_money': my_money,  # 変数名はそのままにしていますが、中身は自分の所持金です
            'wheat_price': market_prices.get('WHEAT', 0),
            'melon_price': market_prices.get('MELON', 0),
            'wheat_in_shed': my_shed.get('WHEAT', 0),
            'melon_in_shed': my_shed.get('MELON', 0),
            'market_action': str(my_market_action) if my_market_action else ""
        }

        data_list.append(row)

    df = pd.DataFrame(data_list)

    print(f"""

分析用json file【{file}】column一覧です



{df.columns.tolist()}

{df.columns.tolist()[0]}:ステップ数
{df.columns.tolist()[1]}:プレイヤー0の所持金
{df.columns.tolist()[2]}:'WHEAT'の値段
{df.columns.tolist()[3]}:'MELON'の値段
{df.columns.tolist()[4]}:'WHEAT'の小屋の在庫
{df.columns.tolist()[5]}:'MELON'小屋の在庫
{df.columns.tolist()[6]}: 市場での売買・雇用アクション

""")

    print("""


グラフを描画しています...

""")

    cols_to_plot = [
        'p0_money',
        'wheat_price',
        'melon_price',
        'wheat_in_shed',
        'melon_in_shed'
    ]


    sns_line_s(df, x='step', y=cols_to_plot, step=30)

    print(df["market_action"].value_counts())

    final_money = df['p0_money'].iloc[-1]

    print(f"""

最後に残ったお金: {final_money} ＄
========================================================
    """)

    return df



In [5]:
df_1 = (j_log("/kaggle/input/datasets/nagatakengo/kaggriculture-nk/v0.6_log/90190473.json"))
df_2 = (j_log("/kaggle/input/datasets/nagatakengo/kaggriculture-nk/v0.6_log/90192486.json"))
df_3 = (j_log("/kaggle/input/datasets/nagatakengo/kaggriculture-nk/v0.6_log/90193959.json"))
df_4 = (j_log("/kaggle/input/datasets/nagatakengo/kaggriculture-nk/v0.6_log/90195410.json"))
df_5 = (j_log("/kaggle/input/datasets/nagatakengo/kaggriculture-nk/v0.6_log/90196015.json"))
df_6 = (j_log("/kaggle/input/datasets/nagatakengo/kaggriculture-nk/v0.6_log/90203848.json"))
df_7 = (j_log("/kaggle/input/datasets/nagatakengo/kaggriculture-nk/v0.6_log/90208122.json"))
df_8 = (j_log("/kaggle/input/datasets/nagatakengo/kaggriculture-nk/v0.6_log/90236958.json"))
df_9 = (j_log("/kaggle/input/datasets/nagatakengo/kaggriculture-nk/v0.6_log/90249656.json"))
df_10 = (j_log("/kaggle/input/datasets/nagatakengo/kaggriculture-nk/v0.6_log/90283277.json"))

'\ndf_1 = (j_log("/kaggle/input/datasets/nagatakengo/kaggriculture-nk/v0.6_log/90190473.json"))\ndf_2 = (j_log("/kaggle/input/datasets/nagatakengo/kaggriculture-nk/v0.6_log/90192486.json"))\ndf_3 = (j_log("/kaggle/input/datasets/nagatakengo/kaggriculture-nk/v0.6_log/90193959.json"))\ndf_4 = (j_log("/kaggle/input/datasets/nagatakengo/kaggriculture-nk/v0.6_log/90195410.json"))\ndf_5 = (j_log("/kaggle/input/datasets/nagatakengo/kaggriculture-nk/v0.6_log/90196015.json"))\ndf_6 = (j_log("/kaggle/input/datasets/nagatakengo/kaggriculture-nk/v0.6_log/90203848.json"))\ndf_7 = (j_log("/kaggle/input/datasets/nagatakengo/kaggriculture-nk/v0.6_log/90208122.json"))\ndf_8 = (j_log("/kaggle/input/datasets/nagatakengo/kaggriculture-nk/v0.6_log/90236958.json"))\ndf_9 = (j_log("/kaggle/input/datasets/nagatakengo/kaggriculture-nk/v0.6_log/90249656.json"))\ndf_10 = (j_log("/kaggle/input/datasets/nagatakengo/kaggriculture-nk/v0.6_log/90283277.json"))\n'